## Setup Environment

### Import Libraries

In [4]:
import pandas as pd
from meteostat import Point, Hourly
import os
import shapely.geometry
import geopandas as gpd
import osmnx as ox
import signal
import matplotlib.pyplot as plt
import matplotlib as mpl

In [5]:
mpl.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 200,
    "axes.linewidth": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "lines.linewidth": 1.5,
})

### Setup Parameters

In [6]:
push_count_folder = "Input/Push_count_data"
devices_input_file = "Input/features_data.csv"
gdb_path_walkability = "Input/Natl_WI.gdb"
gdb_smart_location_file = "Input/SmartLocationDatabase.gdb"
acs_csv_file = "Input/ACS/nhgis0002_csv/nhgis0002_ds267_20235_blck_grp.csv"
acs_shape_file = "Input/ACS/nhgis0002_shape/nhgis0002_shapefile_tl2023_040_blck_grp_2023/AZ_blck_grp_2023.shp"
tes_shade_file = "Input/TreeEquityScore/az_tes.shp"
pag_poi_file = 'Input/PAG/PagEmp_241028.xlsx'
ped_volume_file = 'Input/Ped_volume/raw_pedestrian_data.csv'
maxview_file = 'Input/Maxview/Processed_Maxview.csv'
queue_times_file = 'Input/maxview_waiting_time/Waiting_times.csv'
device_output_file = "Output/features_data.csv"
old_output_file = "Output/output_combined_data4.csv"
final_output_file = "Output/final_pedestrian_data.csv"

In [7]:
exclude_cols = ['DeviceID', 'DateHour', 'Latitude', 'Longitude', 'geometry']

### Helper Functions

In [8]:
def process_static_data(processing_function, feature_list, **kwargs):
    """
    Handles processing and updating the static attributes of the device dataset.

    - Initializes the devices file from `devices_input_file` if it doesn't exist.
    - Applies `processing_function` to update the dataset with new features.
    - Allows additional arguments (`**kwargs`) to be passed to `processing_function`.
    - Saves the updated dataset back to `device_output_file`.

    Parameters:
    - processing_function (function): Function that modifies the dataset.
    - feature_list (list): List of attributes to extract.
    - **kwargs: Additional arguments to pass to the processing function.

    Returns:
    - None (saves the updated dataset to `device_output_file`).
    """

    if not os.path.exists(device_output_file):
        df = pd.read_csv(devices_input_file)
        print("Initializing devices file")
    else:
        df = pd.read_csv(device_output_file)

    df = processing_function(df, feature_list, **kwargs)

    df.to_csv(device_output_file, index=False)

In [9]:
def remove_columns_from_csv(columns_to_remove):
    """
    Removes specified columns from the data

    Parameters:
    - columns_to_remove (list): List of column names to remove.
    """
    df = pd.read_csv(device_output_file)

    df.drop(columns=[col for col in columns_to_remove if col in df.columns], inplace=True)

    df.to_csv(device_output_file, index=False)

## Static Attributes

### EPA

In [5]:
epa_gdf = gpd.read_file(gdb_smart_location_file, layer='EPA_SLD_Database_V3')
for x in epa_gdf.columns:
    print(x)

GEOID10
GEOID20
STATEFP
COUNTYFP
TRACTCE
BLKGRPCE
CSA
CSA_Name
CBSA
CBSA_Name
CBSA_POP
CBSA_EMP
CBSA_WRK
Ac_Total
Ac_Water
Ac_Land
Ac_Unpr
TotPop
CountHU
HH
P_WrkAge
AutoOwn0
Pct_AO0
AutoOwn1
Pct_AO1
AutoOwn2p
Pct_AO2p
Workers
R_LowWageWk
R_MedWageWk
R_HiWageWk
R_PCTLOWWAGE
TotEmp
E5_Ret
E5_Off
E5_Ind
E5_Svc
E5_Ent
E8_Ret
E8_off
E8_Ind
E8_Svc
E8_Ent
E8_Ed
E8_Hlth
E8_Pub
E_LowWageWk
E_MedWageWk
E_HiWageWk
E_PctLowWage
D1A
D1B
D1C
D1C5_RET
D1C5_OFF
D1C5_IND
D1C5_SVC
D1C5_ENT
D1C8_RET
D1C8_OFF
D1C8_IND
D1C8_SVC
D1C8_ENT
D1C8_ED
D1C8_HLTH
D1C8_PUB
D1D
D1_FLAG
D2A_JPHH
D2B_E5MIX
D2B_E5MIXA
D2B_E8MIX
D2B_E8MIXA
D2A_EPHHM
D2C_TRPMX1
D2C_TRPMX2
D2C_TRIPEQ
D2R_JOBPOP
D2R_WRKEMP
D2A_WRKEMP
D2C_WREMLX
D3A
D3AAO
D3AMM
D3APO
D3B
D3BAO
D3BMM3
D3BMM4
D3BPO3
D3BPO4
D4A
D4B025
D4B050
D4C
D4D
D4E
D5AR
D5AE
D5BR
D5BE
D5CR
D5CRI
D5CE
D5CEI
D5DR
D5DRI
D5DE
D5DEI
D2A_Ranked
D2B_Ranked
D3B_Ranked
D4A_Ranked
NatWalkInd
Region
Households
Workers_1
Residents
Drivers
Vehicles
White
Male
Lowwage
Medwage
Highwage


In [16]:
def add_epa_data(df, feature_list):
    """
    Fetches specified EPA data and merges it into the existing dataset.
    If a feature already exists, its values are updated instead of duplicated.

    Parameters:
    - df (pd.DataFrame): Input dataset
    - feature_list (list): List of EPA features to extract.

    Returns:
    - pd.DataFrame: Updated dataset with EPA features merged (replacing previous values if needed).
    """

    epa_gdf = gpd.read_file(gdb_smart_location_file, layer='EPA_SLD_Database_V3')
    epa_gdf = epa_gdf.to_crs("EPSG:4326")

    available_features = [feature for feature in feature_list if feature in epa_gdf.columns]
    epa_gdf = epa_gdf[available_features + ['geometry']]

    df['geometry'] = df.apply(lambda row: shapely.geometry.Point(row['Longitude'], row['Latitude']), axis=1)
    geo_df = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

    geo_df = gpd.sjoin(geo_df, epa_gdf, how="left", predicate="intersects")

    for feature in feature_list:
        if feature not in geo_df.columns:
            geo_df[feature] = None  

    geo_df = geo_df.drop(columns=['geometry', 'index_right'], errors='ignore')

    for feature in feature_list:
        df[feature] = geo_df[feature] if feature in geo_df.columns else None
        df.loc[df[feature] < 0, feature] = 9999

    return df

In [ ]:
process_static_data(
    processing_function=add_epa_data,
    feature_list=[
    "Ac_Total", "Ac_Unpr", "TotEmp", "E5_Ret", "E5_Off", "E5_Ind", "E5_Svc", "E5_Ent", "E8_Ret",
    "E8_off", "E8_Ind", "E8_Svc", "E8_Ent", "E8_Ed", "E8_Hlth", "E8_Pub", "E_LowWageWk", "E_MedWageWk", "E_HiWageWk",
    "E_PctLowWage", "D1A", "D1B", "D1C", "D1C5_RET", "D1C5_OFF", "D1C5_IND", "D1C5_SVC", "D1C5_ENT", "D1C8_RET",
    "D1C8_OFF", "D1C8_IND", "D1C8_SVC", "D1C8_ENT", "D1C8_ED", "D1C8_HLTH", "D1C8_PUB", "D1D", "D1_FLAG", "D2A_JPHH",
    "D2B_E5MIX", "D2B_E5MIXA", "D2B_E8MIX", "D2B_E8MIXA", "D2A_EPHHM", "D2C_TRPMX1", "D2C_TRPMX2", "D2C_TRIPEQ",
    "D2R_JOBPOP", "D2R_WRKEMP", "D2A_WRKEMP", "D2C_WREMLX", "D3A", "D3AAO", "D3AMM", "D3APO", "D3B", "D3BAO", "D3BMM3",
    "D3BMM4", "D3BPO3", "D3BPO4", "D4A", "D4B025", "D4B050", "D4C", "D4D", "D4E", "D5AR", "D5AE", "D5BR", "D5BE", "D5CR",
    "D5CRI", "D5CE", "D5CEI", "D5DR", "D5DRI", "D5DE", "D5DEI", "D2A_Ranked", "D2B_Ranked", "D3B_Ranked", "D4A_Ranked",
    "NatWalkInd"]
)

### OSM POI

In [164]:
import os
import time
import pandas as pd
import osmnx as ox
from shapely.geometry import Point
from concurrent.futures import ThreadPoolExecutor, TimeoutError

poi_tags = {
    'amenity': ['fast_food', 'parking', 'theatre', 'school', 'fuel', 'healthcare'],
    'shop': True,
    'leisure': True,
    'tourism': True
}

def fetch_poi_counts(lat, lon, dist=402):
    pois = ox.features_from_point((lat, lon), tags=poi_tags, dist=dist)
    return {
        'Fast_food': len(pois[pois['amenity'] == 'fast_food']) if 'amenity' in pois else 0,
        'Parking': len(pois[pois['amenity'] == 'parking']) if 'amenity' in pois else 0,
        'Theatre': len(pois[pois['amenity'] == 'theatre']) if 'amenity' in pois else 0,
        'School': len(pois[pois['amenity'] == 'school']) if 'amenity' in pois else 0,
        'Fuel': len(pois[pois['amenity'] == 'fuel']) if 'amenity' in pois else 0,
        'Shops': len(pois[pois['shop'].notna()]) if 'shop' in pois else 0,
        'Leisure': len(pois[pois['leisure'].notna()]) if 'leisure' in pois else 0,
        'Tourism': len(pois[pois['tourism'].notna()]) if 'tourism' in pois else 0,
        'Healthcare': len(pois[pois['amenity'] == 'healthcare']) if 'amenity' in pois else 0,
    }

def add_osm_poi_data(df, feature_list=None, radius_m=402, retry_wait=5, timeout_sec=15, temp_output_path=None):
    df = df.copy()

    if temp_output_path and os.path.exists(temp_output_path):
        existing = pd.read_csv(temp_output_path)
        results = existing.to_dict("records")
        completed_set = set(zip(existing['Latitude'], existing['Longitude']))
    else:
        results = []
        completed_set = set()

    for idx, row in df.drop_duplicates(subset='DeviceID').iterrows():
        number_of_tries = 0
        lat, lon, id = row['Latitude'], row['Longitude'], row['DeviceID']
        loc_key = (lat, lon)

        if pd.isnull(lat) or pd.isnull(lon) or loc_key in completed_set:
            continue

        print(f"Processing {idx}: ({lat}, {lon})")

        success = False
        while not success:
            try:
                number_of_tries += 1
                with ThreadPoolExecutor(max_workers=1) as executor:
                    future = executor.submit(fetch_poi_counts, lat, lon, radius_m)
                    counts = future.result(timeout=timeout_sec)

                result_row = {'Latitude': lat, 'Longitude': lon, 'DeviceID': id}
                result_row.update(counts)
                results.append(result_row)

                # ✅ Save to temp CSV after every successful row
                if temp_output_path:
                    pd.DataFrame(results).to_csv(temp_output_path, index=False)

                success = True
                print(f"✓ Completed index {idx}")

            except TimeoutError:
                print(f"Timeout at index {idx}. Retrying in {retry_wait} seconds...")
                time.sleep(retry_wait)
            except Exception as e:
                if number_of_tries >= 3:
                    result_row = {'Latitude': lat, 'Longitude': lon, 'DeviceID': id}
                    result_row.update({
                        'Fast_food': 0,
                        'Parking': 0,
                        'Theatre': 0,
                        'School': 0,
                        'Fuel': 0,
                        'Shops':  0,
                        'Leisure': 0,
                        'Tourism': 0,
                        'Healthcare': 0,
                    })
                    results.append(result_row)

                    if temp_output_path:
                        pd.DataFrame(results).to_csv(temp_output_path, index=False)

                    success = True
                    print(f"✓ Completed index {idx}")
                else:
                    print(f"Error at index {idx}: {e}. Retrying in {retry_wait} seconds...")
                    time.sleep(retry_wait)

    if results:
        results_df = pd.DataFrame(results)
        results_df = results_df.drop(columns=['Latitude', 'Longitude'], errors='ignore')
        df = pd.merge(df, results_df, on='DeviceID', how='inner').reset_index(drop=True)

    return df

In [165]:
process_static_data(
    processing_function=add_osm_poi_data,
    feature_list=[],
    temp_output_path = "osm_poi_checkpoint.csv"
)

In [30]:
import os
import time
import pandas as pd
import geopandas as gpd
import shapely.geometry
import osmnx as ox
from concurrent.futures import ThreadPoolExecutor, TimeoutError

RADIUS_M = 402  # quarter-mile in meters
CHECKPOINT_FILE = "osm_cycleway_checkpoint.csv"
TIMEOUT = 20  # seconds

def get_local_utm_crs(lat, lon):
    zone = int((lon + 180) / 6) + 1
    hemisphere = 'north' if lat >= 0 else 'south'
    return f"+proj=utm +zone={zone} +{hemisphere} +ellps=WGS84 +datum=WGS84 +units=m +no_defs"

def fetch_roads(bbox):
    return ox.features_from_bbox(
        north=bbox[3], south=bbox[1], east=bbox[2], west=bbox[0], tags={"highway": True}
    )

def add_osm_geometry_data(df, feature_list=None):
    df = df.copy()
    df['geometry'] = df.apply(lambda row: shapely.geometry.Point(row['Longitude'], row['Latitude']), axis=1)

    if os.path.exists(CHECKPOINT_FILE):
        checkpoint_df = pd.read_csv(CHECKPOINT_FILE)
        completed_indices = set(checkpoint_df['index'])
        pct_cycleways = checkpoint_df.set_index('index')['Pct_cycleways_in_radius'].to_dict()
        print(f"Resuming from checkpoint. {len(completed_indices)} already completed.")
    else:
        completed_indices = set()
        pct_cycleways = {}

    total = len(df)
    for i, row in df.iterrows():
        if i in completed_indices:
            continue

        lat = row['Latitude']
        lon = row['Longitude']
        try:
            point = shapely.geometry.Point(lon, lat)
            gdf_pt = gpd.GeoDataFrame(index=[0], geometry=[point], crs='EPSG:4326')
            utm_crs = get_local_utm_crs(lat, lon)
            gdf_proj = gdf_pt.to_crs(utm_crs)
            buffer = gdf_proj.buffer(RADIUS_M).iloc[0]

            buffer_wgs = gpd.GeoSeries([buffer], crs=utm_crs).to_crs('EPSG:4326').iloc[0]
            minx, miny, maxx, maxy = buffer_wgs.bounds
            bbox = (minx, miny, maxx, maxy)

            roads = None
            with ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(fetch_roads, bbox)
                try:
                    roads = future.result(timeout=TIMEOUT)
                except TimeoutError:
                    raise TimeoutError(f"Timeout while querying OSM for index {i}")

            if roads is None or roads.empty:
                pct_cycleways[i] = None
                print(f"{i}/{total} – ({lat:.4f}, {lon:.4f}): No roads found.")
                continue

            roads = roads[roads.geometry.type.isin(['LineString', 'MultiLineString'])]
            roads = roads[~roads.geometry.is_empty]
            roads_proj = roads.to_crs(utm_crs)
            clipped = gpd.clip(roads_proj, buffer)
            if clipped.empty:
                pct_cycleways[i] = None
                print(f"{i}/{total} – ({lat:.4f}, {lon:.4f}): No clipped roads.")
                continue

            clipped['length_m'] = clipped.geometry.length
            clipped['has_cycleway'] = clipped.apply(
                lambda r: any("cycleway" in str(k).lower() and pd.notna(v) for k, v in r.items()),
                axis=1
            )

            total_len = clipped['length_m'].sum()
            cycle_len = clipped.loc[clipped['has_cycleway'], 'length_m'].sum()
            pct = cycle_len / total_len if total_len > 0 else None
            pct_cycleways[i] = pct

        except Exception as e:
            print(f"Error at {i} ({lat:.4f}, {lon:.4f}): {e}")
            pct_cycleways[i] = None

        pd.DataFrame({
            'index': list(pct_cycleways.keys()),
            'Pct_cycleways_in_radius': list(pct_cycleways.values())
        }).to_csv(CHECKPOINT_FILE, index=False)

        print(f"✓ {i}/{total} – ({lat:.4f}, {lon:.4f}) – {pct_cycleways[i]}")

    df['Pct_cycleways_in_radius'] = [pct_cycleways.get(i, None) for i in df.index]
    return df

In [31]:
process_static_data(
    processing_function=add_osm_geometry_data,
    feature_list=[]
)

Resuming from checkpoint. 41 already completed.


In [116]:
import requests

def get_all_osm_tags(osmid, osm_type='way'):
    """
    Queries Overpass API for all tags of a given OSM ID.
    
    Args:
        osmid (int): The OSM ID of the feature.
        osm_type (str): One of 'node', 'way', or 'relation'. Default is 'way'.
        
    Returns:
        dict: A dictionary of all OSM tags.
    """
    overpass_url = "http://overpass-api.de/api/interpreter"
    query = f"""
    [out:json];
    {osm_type}({osmid});
    out tags;
    """
    response = requests.get(overpass_url, params={'data': query})
    if response.status_code == 200:
        data = response.json()
        if 'elements' in data and len(data['elements']) > 0:
            return data['elements'][0].get('tags', {})
        else:
            print(f"No data found for {osm_type} {osmid}")
            return {}
    else:
        print(f"Failed to query Overpass API for {osm_type} {osmid}: HTTP {response.status_code}")
        return {}

In [123]:
import os
import time
import pandas as pd
import osmnx as ox
from shapely.geometry import Point
from concurrent.futures import ThreadPoolExecutor, TimeoutError

# Fallback speed limits by highway type (in mph)
DEFAULT_SPEEDS = {
    'motorway': 65,
    'trunk': 55,
    'primary': 45,
    'secondary': 35,
    'tertiary': 30,
    'residential': 25,
    'unclassified': 25,
    'service': 15
}

# Fallback lane count by highway type
DEFAULT_LANES = {
    'motorway': 4,
    'trunk': 3,
    'primary': 2,
    'secondary': 2,
    'tertiary': 2,
    'residential': 2,
    'unclassified': 1,
    'service': 1
}

DEFAULT_WIDTH_PER_LANE = 3.65

def fetch_nearest_road_info(lat, lon, dist=402):
    lat = float(lat)
    lon = float(lon)

    G = ox.graph_from_point((lat, lon), dist=400, network_type='drive', simplify=False, retain_all=True)
    u, v, k = ox.distance.nearest_edges(G, lon, lat)
    data = G[u][v][k]
    
    osmid = data.get('osmid')
    if isinstance(osmid, list):
        osmid = osmid[0]

    all_tags = get_all_osm_tags(osmid, osm_type='way')

    highway_type = data.get('highway')
    if isinstance(highway_type, list):
        highway_type = highway_type[0]

    has_cycleway = 'cycleway' in data or 'cycleway' in all_tags
    has_sidewalk = 'sidewalk' in data or 'sidewalk' in all_tags

    # Speed limit
    speed_val = data.get('maxspeed') or all_tags.get('maxspeed')
    speed = None
    if speed_val:
        if isinstance(speed_val, list):
            speed_val = speed_val[0]
        try:
            speed_str = str(speed_val).lower()
            if 'mph' in speed_str:
                speed = float(speed_str.replace('mph', '').strip())
            elif 'km' in speed_str:
                speed = float(speed_str.replace('km/h', '').strip()) * 0.621371
            else:
                speed = float(speed_str)
        except:
            pass
    if speed is None:
        speed = DEFAULT_SPEEDS.get(highway_type, None)

    width_val = data.get('width') or all_tags.get('width')
    lanes_val = data.get('lanes') or all_tags.get('lanes')
    lanes = None
    try:
        if isinstance(lanes_val, list):
            lanes_val = lanes_val[0]
        lanes = int(str(lanes_val))
    except:
        lanes = DEFAULT_LANES.get(highway_type, 1)
    width = float(width_val) if width_val else lanes * DEFAULT_WIDTH_PER_LANE

    return {
        'osmid': osmid,
        'Has_cycleway_on_nearest': has_cycleway,
        'Has_sidewalk_on_nearest': has_sidewalk,
        'Max_speed_on_nearest': round(speed, 1) if speed else None,
        'Estimated_width_m': round(width, 1),
        'Road_name': data.get('name') or all_tags.get('name'),
    }

def add_osm_road_info(df, radius_m=402, retry_wait=5, timeout_sec=15, temp_output_path=None):
    df = df.copy()
    if temp_output_path and os.path.exists(temp_output_path):
        existing = pd.read_csv(temp_output_path)
        completed_set = set(zip(existing['Latitude'], existing['Longitude']))
        results = existing.to_dict("records")
    else:
        completed_set = set()
        results = []

    for idx, row in df.iterrows():
        lat, lon = row['Latitude'], row['Longitude']
        loc_key = (lat, lon)

        if pd.isnull(lat) or pd.isnull(lon) or loc_key in completed_set:
            continue

        print(f"Processing {idx}: ({lat}, {lon})")

        success = False
        while not success:
            try:
                with ThreadPoolExecutor(max_workers=1) as executor:
                    future = executor.submit(fetch_nearest_road_info, lat, lon, radius_m)
                    road_info = future.result(timeout=timeout_sec)

                result_row = {'Latitude': lat, 'Longitude': lon}
                result_row.update(road_info)
                results.append(result_row)

                if temp_output_path:
                    pd.DataFrame(results).to_csv(temp_output_path, index=False)

                print(f"✓ Completed index {idx}")
                success = True

            except TimeoutError:
                print(f"Timeout at index {idx}. Retrying in {retry_wait}s...")
                time.sleep(retry_wait)
            except Exception as e:
                print(f"Error at index {idx}: {e}. Retrying in {retry_wait}s...")
                time.sleep(retry_wait)

    if results:
        new_data = pd.DataFrame(results)
        new_data = new_data.drop(columns=['Latitude', 'Longitude'], errors='ignore')
        df = pd.concat([df.reset_index(drop=True), new_data.reset_index(drop=True)], axis=1)

    return df

In [124]:
process_static_data(
    processing_function=add_osm_road_info,
    feature_list=[],
    temp_output_path = "osm_roadway_checkpoint.csv"
)

Processing 0: (32.2263, -110.9987)


✓ Completed index 0
Processing 1: (32.2251, -110.9892)


✓ Completed index 1
Processing 2: (32.225075, -110.982619)


✓ Completed index 2
Processing 3: (32.2198, -110.9852)


✓ Completed index 3
Processing 4: (32.2177, -110.9844)


...

✓ Completed index 36
Processing 37: (32.2196, -110.9563)


✓ Completed index 37
Processing 38: (32.2236, -110.9739)


✓ Completed index 38
Processing 39: (32.2330879, -110.9657828)


✓ Completed index 39
Processing 40: (32.222249, -110.988458)


✓ Completed index 40


### OSM Road Data

In [63]:
import os
import time
import pandas as pd
import osmnx as ox
from shapely.geometry import Point
from concurrent.futures import ThreadPoolExecutor, TimeoutError
import requests
import geopandas as gpd
from shapely import affinity
import numpy as np
import math
import warnings

warnings.filterwarnings('ignore')

def get_all_osm_tags(osmid, osm_type='way'):
    overpass_url = "http://overpass-api.de/api/interpreter"
    query = f"""
    [out:json];
    {osm_type}({osmid});
    out tags;
    """
    response = requests.get(overpass_url, params={'data': query})
    if response.status_code == 200:
        data = response.json()
        if 'elements' in data and len(data['elements']) > 0:
            return data['elements'][0].get('tags', {})
        else:
            return {}
    else:
        return {}

DEFAULT_SPEEDS = {
    'motorway': 65, 'trunk': 55, 'primary': 45, 'secondary': 35,
    'tertiary': 30, 'residential': 25, 'unclassified': 25, 'service': 15
}
DEFAULT_LANES = {
    'motorway': 6, 'trunk': 4, 'primary': 4, 'secondary': 4,
    'tertiary': 2, 'residential': 2, 'unclassified': 2, 'service': 1
}
DEFAULT_WIDTH_PER_LANE = 3.65

def has_key_with_substring(d, substring):
    return any(substring in k.lower() for k in d.keys())

def extract_segment_info(segment):
    osmid = segment.get('osmid')
    if isinstance(osmid, list): osmid = osmid[0]
    all_tags = get_all_osm_tags(osmid)

    highway_type = segment.get('highway')
    if isinstance(highway_type, list): highway_type = highway_type[0]

    has_sidewalk = has_key_with_substring(segment, 'sidewalk') or has_key_with_substring(all_tags, 'sidewalk')
    has_cycleway = has_key_with_substring(segment, 'cycleway') or has_key_with_substring(all_tags, 'cycleway')

    speed_val = segment.get('maxspeed') or all_tags.get('maxspeed')
    if isinstance(speed_val, list): speed_val = speed_val[0]
    try:
        speed_str = str(speed_val).lower()
        if 'mph' in speed_str:
            speed = float(speed_str.replace('mph', '').strip())
        elif 'km' in speed_str:
            speed = float(speed_str.replace('km/h', '').strip()) * 0.621371
        else:
            speed = float(speed_str)
    except:
        speed = DEFAULT_SPEEDS.get(highway_type, 25)

    width_val = segment.get('width') or all_tags.get('width')
    lanes_val = segment.get('lanes') or all_tags.get('lanes')
    if isinstance(lanes_val, list): lanes_val = lanes_val[0]
    try:
        lanes = int(str(lanes_val))
    except:
        lanes = DEFAULT_LANES.get(highway_type, 2)
    width = float(width_val) if width_val else lanes * DEFAULT_WIDTH_PER_LANE

    return {
        'speed': speed,
        'width': width,
        'has_sidewalk': has_sidewalk,
        'has_cycleway': has_cycleway
    }

def summarize_direction_info(segments):
    speeds, widths, sidewalks, cycleways = [], [], [], []
    for _, segment in segments.iterrows():
        info = extract_segment_info(segment)
        if info['speed'] is not None: speeds.append(info['speed'])
        widths.append(info['width'])
        sidewalks.append(info['has_sidewalk'])
        cycleways.append(info['has_cycleway'])

    return {
        'Max_speed_mph': round(min(speeds), 1) if speeds else 25,
        'Total_width_m': round(sum(widths), 1),
        'Has_sidewalk': any(sidewalks),
        'Has_cycleway': any(cycleways)
    }

def fetch_and_analyze_segments(lat, lon):
    pt = Point(lon, lat)
    G = ox.graph_from_point((lat, lon), dist=40, network_type='drive', simplify=False)
    edges = ox.graph_to_gdfs(G, nodes=False).to_crs(epsg=3857)
    pt_proj = gpd.GeoSeries([pt], crs='EPSG:4326').to_crs(epsg=3857).iloc[0]
    buffer = pt_proj.buffer(40)
    segments = edges[edges.intersects(buffer)].copy().reset_index(drop=True)
    segments["length"] = segments["geometry"].length

    def segment_angle(geom):
        x1, y1 = geom.coords[0]
        x2, y2 = geom.coords[-1]
        return math.degrees(math.atan2(y2 - y1, x2 - x1)) % 180

    angle_list = [segment_angle(row.geometry) for _, row in segments.iterrows() if row['length'] > 15]
    hori_angles = [a for a in angle_list if a < 45 or a > 135]
    mean_angle = np.mean([a if a < 90 else a - 180 for a in hori_angles]) if hori_angles else 0

    rotated_segments = segments.copy()
    rotated_segments['geometry'] = rotated_segments['geometry'].apply(
        lambda geom: affinity.rotate(geom, -mean_angle, origin=pt_proj, use_radians=False)
    )

    vertical, horizontal = [], []
    for i, row in rotated_segments.iterrows():
        x1, y1 = row.geometry.coords[0]
        x2, y2 = row.geometry.coords[-1]
        if (abs(y2 - y1) > 0) and abs(x2 - x1) / abs(y2 - y1) < 1:
            c = (x1 + x2) / 2
            vertical.append((i, c, row.length))
        else:
            c = (y1 + y2) / 2
            horizontal.append((i, c, row.length))

    def group_by_axis(lines, threshold=10):
        groups, used = [], set()
        for i, c1, _ in lines:
            if i in used: continue
            group = [i]
            for j, c2, _ in lines:
                if j != i and j not in used and abs(c1 - c2) < threshold:
                    group.append(j)
            used.update(group)
            groups.append(group)
        return groups

    vert_groups = group_by_axis(vertical)
    hori_groups = group_by_axis(horizontal)
    selected_verticals = [segments.loc[group].sort_values("length", ascending=False).iloc[0] for group in vert_groups]
    selected_horizontals = [segments.loc[group].sort_values("length", ascending=False).iloc[0] for group in hori_groups]

    vertical_df = pd.DataFrame(selected_verticals)
    horizontal_df = pd.DataFrame(selected_horizontals)

    return {
        'NS': summarize_direction_info(vertical_df),
        'EW': summarize_direction_info(horizontal_df)
    }

def get_local_utm_crs(lat, lon):
    zone = int((lon + 180) / 6) + 1
    hemisphere = 'north' if lat >= 0 else 'south'
    return f"+proj=utm +zone={zone} +{hemisphere} +ellps=WGS84 +datum=WGS84 +units=m +no_defs"

def fetch_roads(bbox):
    return ox.features_from_bbox(
        north=bbox[3], south=bbox[1], east=bbox[2], west=bbox[0], tags={"highway": True}
    )

def compute_pct_facilities(lat, lon, radius_m=402, timeout=20):
    point = shapely.geometry.Point(lon, lat)
    gdf_pt = gpd.GeoDataFrame(index=[0], geometry=[point], crs='EPSG:4326')
    utm_crs = get_local_utm_crs(lat, lon)
    gdf_proj = gdf_pt.to_crs(utm_crs)
    buffer = gdf_proj.buffer(radius_m).iloc[0]

    buffer_wgs = gpd.GeoSeries([buffer], crs=utm_crs).to_crs('EPSG:4326').iloc[0]
    minx, miny, maxx, maxy = buffer_wgs.bounds
    bbox = (minx, miny, maxx, maxy)

    roads = None
    with ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(lambda: ox.features_from_bbox(
            north=bbox[3], south=bbox[1], east=bbox[2], west=bbox[0], tags={"highway": True}))
        try:
            roads = future.result(timeout=timeout)
        except TimeoutError:
            return None, None

    if roads is None or roads.empty:
        return None, None

    roads = roads[roads.geometry.type.isin(['LineString', 'MultiLineString'])]
    roads = roads[~roads.geometry.is_empty]
    roads_proj = roads.to_crs(utm_crs)
    clipped = gpd.clip(roads_proj, buffer)
    if clipped.empty:
        return None, None

    clipped['length_m'] = clipped.geometry.length
    clipped['has_cycleway'] = clipped.apply(
        lambda r: any("cycleway" in str(k).lower() and pd.notna(v) for k, v in r.items()),
        axis=1
    )
    clipped['has_sidewalk'] = clipped.apply(
        lambda r: any("sidewalk" in str(k).lower() and pd.notna(v) for k, v in r.items()),
        axis=1
    )

    total_len = clipped['length_m'].sum()
    pct_cycle = clipped.loc[clipped['has_cycleway'], 'length_m'].sum() / total_len if total_len > 0 else None
    pct_side = clipped.loc[clipped['has_sidewalk'], 'length_m'].sum() / total_len if total_len > 0 else None
    return pct_cycle, pct_side

def add_osm_road_data(df, retry_wait=5, timeout_sec=30, temp_output_path=None):
    df = df.copy()

    if temp_output_path and os.path.exists(temp_output_path):
        existing = pd.read_csv(temp_output_path)
        results = existing.to_dict("records")
        completed_set = set(zip(existing['Latitude'], existing['Longitude']))
    else:
        results = []
        completed_set = set()

    for idx, row in df.drop_duplicates(subset='DeviceID').iterrows():
        number_of_tries = 0
        lat, lon, device_id = row['Latitude'], row['Longitude'], row['DeviceID']
        loc_key = (lat, lon)
        if pd.isnull(lat) or pd.isnull(lon) or loc_key in completed_set:
            continue

        print(f"Processing {idx}: ({lat}, {lon})")
        success = False
        start_time = time.time()

        while not success and (time.time() - start_time) < 3:
            try:
                number_of_tries += 1
                with ThreadPoolExecutor(max_workers=1) as executor:
                    future = executor.submit(fetch_and_analyze_segments, lat, lon)
                    summary = future.result(timeout=timeout_sec)

                result_row = {'Latitude': lat, 'Longitude': lon, 'DeviceID': device_id}
                for prefix, info in summary.items():
                    result_row.update({f'{prefix}_{k}': v for k, v in info.items()})

                pct_cycle, pct_side = compute_pct_facilities(lat, lon)
                result_row['Pct_cycleways_in_radius'] = pct_cycle
                result_row['Pct_sidewalks_in_radius'] = pct_side
                results.append(result_row)

                if temp_output_path:
                    pd.DataFrame(results).to_csv(temp_output_path, index=False)

                print(f"✓ Completed index {idx}")
                success = True

                if number_of_tries >= 3:
                    break

            except TimeoutError:
                print(f"Timeout at index {idx}. Retrying in {retry_wait}s...")
                time.sleep(retry_wait)
            except Exception as e:
                print(f"Error at index {idx}: {e}. Retrying in {retry_wait}s...")
                time.sleep(retry_wait)

        if not success:
            print(f"✗ Giving up on index {idx} after 3 seconds.")
            result_row = {'Latitude': lat, 'Longitude': lon, 'DeviceID': device_id}
            result_row.update({
                'NS_Max_speed_mph': 25,
                'NS_Total_width_m': 2 * DEFAULT_WIDTH_PER_LANE,
                'NS_Has_sidewalk': False,
                'NS_Has_cycleway': False,
                'EW_Max_speed_mph': 25,
                'EW_Total_width_m': 2 * DEFAULT_WIDTH_PER_LANE,
                'EW_Has_sidewalk': False,
                'EW_Has_cycleway': False,
                'Pct_cycleways_in_radius': 0,
                'Pct_sidewalks_in_radius': 0
            })
            results.append(result_row)
            if temp_output_path:
                pd.DataFrame(results).to_csv(temp_output_path, index=False)

    if results:
        results_df = pd.DataFrame(results)
        results_df = results_df.drop(columns=['Latitude', 'Longitude'], errors='ignore')
        df = pd.merge(df, results_df, on='DeviceID', how='inner').reset_index(drop=True)

    return df

In [68]:
process_static_data(
    processing_function=add_osm_road_data,
    feature_list=[],
    temp_output_path="osm_roadway_checkpoint.csv"
)

### TES

In [130]:
import geopandas as gpd
import pandas as pd
import shapely.geometry

def add_tes_data(df, feature_list):
    """
    Fetches specified Tree Equity Score (TES) data and merges it into the existing dataset.
    If a feature already exists, its values are updated instead of duplicated.

    Parameters:
    - df (pd.DataFrame): Input dataset containing at least 'Latitude' and 'Longitude' columns.
    - feature_list (list): List of TES features (columns) to extract.
    - tes_shapefile_path (str): Path to the TES shapefile (.shp or .gpkg).

    Returns:
    - pd.DataFrame: Updated dataset with TES features merged (replacing previous values if needed).
    """

    tes_gdf = gpd.read_file(tes_shade_file)
    tes_gdf = tes_gdf.to_crs("EPSG:4326")

    available_features = [feature for feature in feature_list if feature in tes_gdf.columns]
    tes_gdf = tes_gdf[available_features + ['geometry']]

    df['geometry'] = df.apply(lambda row: shapely.geometry.Point(row['Longitude'], row['Latitude']), axis=1)
    geo_df = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

    geo_df = gpd.sjoin(geo_df, tes_gdf, how="left", predicate="within")

    for feature in feature_list:
        if feature not in geo_df.columns:
            geo_df[feature] = None

    geo_df = geo_df.drop(columns=['geometry', 'index_right'], errors='ignore')

    for feature in feature_list:
        df[feature] = geo_df[feature] if feature in geo_df.columns else None

    return df

In [131]:
process_static_data(
    processing_function=add_tes_data,
    feature_list=['treecanopy', 'tc_goal', 'priority_i', 'temp_diff']
)

### ACS

In [8]:
epa_gdf = gpd.read_file(acs_shape_file, layer='AZ_blck_grp_2023')
for x in epa_gdf.columns:
    print(x)

GISJOIN
STATEFP
COUNTYFP
TRACTCE
BLKGRPCE
GEOID
GEOIDFQ
NAMELSAD
MTFCC
FUNCSTAT
ALAND
AWATER
Shape_Leng
Shape_Area
ORIG_FID
geometry


In [51]:
def add_acs_data(df, feature_list):
    """
    Fetches specified ACS data by combining shapefile and CSV, then merges selected features
    into the existing dataset using spatial join. Existing features are replaced if present.

    Parameters:
    - df (pd.DataFrame): Input dataset with 'Latitude' and 'Longitude'
    - feature_list (list): List of dicts mapping original ACS feature → desired name

    Returns:
    - pd.DataFrame: Updated dataset with ACS features merged (replacing values if needed)
    """

    orig_cols = set(df.columns)

    acs_csv_df = pd.read_csv(acs_csv_file, dtype={'GISJOIN': str})
    acs_gdf = gpd.read_file(acs_shape_file, layer='AZ_blck_grp_2023')
    acs_gdf['GISJOIN'] = acs_gdf['GISJOIN'].astype(str)

    acs_gdf = acs_gdf.merge(acs_csv_df, on='GISJOIN', how='inner')

    acs_gdf = acs_gdf.to_crs(epsg=3857)  # Work in meters
    acs_gdf['bg_area_m2'] = acs_gdf.geometry.area
    acs_gdf['area_km2'] = acs_gdf['bg_area_m2'] / 1e6
    acs_gdf['Total_pop'] = acs_gdf['ASN2E001']
    acs_gdf['Pop_density'] = acs_gdf['ASN2E001'] / acs_gdf['area_km2']

    working_age_cols = [
        'ASNQE007', 'ASNQE008', 'ASNQE009', 'ASNQE010', 'ASNQE011', 'ASNQE012', 'ASNQE013',
        'ASNQE014', 'ASNQE015', 'ASNQE016', 'ASNQE017', 'ASNQE018', 'ASNQE019',
        'ASNQE020', 'ASNQE021', 'ASNQE022', 'ASNQE023', 'ASNQE024', 'ASNQE025',
        'ASNQE031', 'ASNQE032', 'ASNQE033', 'ASNQE034', 'ASNQE035', 'ASNQE036', 'ASNQE037',
        'ASNQE038', 'ASNQE039', 'ASNQE040', 'ASNQE041', 'ASNQE042', 'ASNQE043',
        'ASNQE044', 'ASNQE045', 'ASNQE046', 'ASNQE047', 'ASNQE048', 'ASNQE049'
    ]

    total_working_age = acs_gdf[working_age_cols].sum(axis=1)

    acs_gdf['Pct_working_age'] = total_working_age / acs_gdf['Total_pop']

    child_cols = [
        'ASNQE003', 'ASNQE004', 'ASNQE005', 'ASNQE006',  # Male under 18
        'ASNQE027', 'ASNQE028', 'ASNQE029', 'ASNQE030'   # Female under 18
    ]
    acs_gdf['Child_dependency_ratio'] = acs_gdf[child_cols].sum(axis=1) / total_working_age

    elderly_cols = [
        'ASNQE020', 'ASNQE021', 'ASNQE022', 'ASNQE023', 'ASNQE024', 'ASNQE025',
        'ASNQE044', 'ASNQE045', 'ASNQE046', 'ASNQE047', 'ASNQE048', 'ASNQE049'
    ]
    acs_gdf['Elderly_dependency_ratio'] = acs_gdf[elderly_cols].sum(axis=1) / total_working_age

    acs_gdf['Sex_ratio'] = acs_gdf['ASNQE002'] / acs_gdf['ASNQE026']

    school_age_cols = [
        'ASNQE004', 'ASNQE005', 'ASNQE006',  # Male: 5–17
        'ASNQE028', 'ASNQE029', 'ASNQE030'   # Female: 5–17
    ]
    acs_gdf['Pct_school_age'] = acs_gdf[school_age_cols].sum(axis=1) / acs_gdf['Total_pop']

    acs_gdf['White_to_others_ratio'] = acs_gdf['ASN2E002'] / (acs_gdf['ASN2E001'] - acs_gdf['ASN2E002'])

    acs_gdf['Pct_drive_alone'] = acs_gdf['ASORE003'] / acs_gdf['ASORE001']
    acs_gdf['Pct_carpool'] = acs_gdf['ASORE004'] / acs_gdf['ASORE001']
    acs_gdf['Pct_public_transit'] = acs_gdf['ASORE010'] / acs_gdf['ASORE001']
    acs_gdf['Pct_walk'] = acs_gdf['ASORE019'] / acs_gdf['ASORE001']
    acs_gdf['Pct_bike'] = acs_gdf['ASORE018'] / acs_gdf['ASORE001']

    acs_gdf['Pct_early_morning'] = acs_gdf['ASOSE002'] / acs_gdf['ASOSE001']  # 12:00–4:59 AM
    peak_cols = ['ASOSE005', 'ASOSE006', 'ASOSE007', 'ASOSE008', 'ASOSE009']
    acs_gdf['Pct_peak_departure'] = acs_gdf[peak_cols].sum(axis=1) / acs_gdf['ASOSE001']
    acs_gdf['Pct_evening'] = acs_gdf['ASOSE015'] / acs_gdf['ASOSE001']

    short_cols = ['ASOTE002', 'ASOTE003', 'ASOTE004']
    acs_gdf['Pct_short_commute'] = acs_gdf[short_cols].sum(axis=1) / acs_gdf['ASOTE001']
    moderate_cols = ['ASOTE005', 'ASOTE006', 'ASOTE007']
    acs_gdf['Pct_moderate_commute'] = acs_gdf[moderate_cols].sum(axis=1) / acs_gdf['ASOTE001']
    long_cols = ['ASOTE008', 'ASOTE009', 'ASOTE010', 'ASOTE011', 'ASOTE012', 'ASOTE013']
    acs_gdf['Pct_long_commute'] = acs_gdf[long_cols].sum(axis=1) / acs_gdf['ASOTE001']
    super_commute_cols = ['ASOTE012', 'ASOTE013']
    acs_gdf['Pct_super_commuters'] = acs_gdf[super_commute_cols].sum(axis=1) / acs_gdf['ASOTE001']

    hs_or_higher_cols = [
        'ASP3E017', 'ASP3E018', 'ASP3E019', 'ASP3E020',
        'ASP3E021', 'ASP3E022', 'ASP3E023', 'ASP3E024', 'ASP3E025'
    ]
    acs_gdf['Pct_hs_or_higher'] = acs_gdf[hs_or_higher_cols].sum(axis=1) / acs_gdf['ASP3E001']

    bachelors_or_higher_cols = ['ASP3E022', 'ASP3E023', 'ASP3E024', 'ASP3E025']
    acs_gdf['Pct_bachelors_or_higher'] = acs_gdf[bachelors_or_higher_cols].sum(axis=1) / acs_gdf['ASP3E001']

    less_than_hs_cols = [
        'ASP3E002', 'ASP3E003', 'ASP3E004', 'ASP3E005', 'ASP3E006', 'ASP3E007', 'ASP3E008',
        'ASP3E009', 'ASP3E010', 'ASP3E011', 'ASP3E012', 'ASP3E013', 'ASP3E014', 'ASP3E015',
        'ASP3E016'
    ]
    acs_gdf['Pct_less_than_hs'] = acs_gdf[less_than_hs_cols].sum(axis=1) / acs_gdf['ASP3E001']

    acs_gdf['Pct_below_poverty'] = (
        acs_gdf[['ASQIE002', 'ASQIE003']].sum(axis=1) / acs_gdf['ASQIE001']
    )
    acs_gdf['Pct_near_poverty'] = (
        acs_gdf[['ASQIE004', 'ASQIE005', 'ASQIE006', 'ASQIE007']].sum(axis=1) / acs_gdf['ASQIE001']
    )
    acs_gdf['Pct_above_2x_poverty'] = acs_gdf['ASQIE008'] / acs_gdf['ASQIE001']

    acs_gdf['Per_capita_income'] = acs_gdf['ASRTE001']

    acs_gdf['Housing_unit_density'] = acs_gdf['ASS7E001'] / acs_gdf['area_km2']

    acs_gdf['Pct_occupied_houses'] = acs_gdf['ASS8E002'] / acs_gdf['ASS8E001']

    acs_gdf['Pct_no_vehicle'] = (
        acs_gdf[['ASUTE003', 'ASUTE010']].sum(axis=1) / acs_gdf['ASUTE001']
    )
    acs_gdf['Pct_one_vehicle'] = (
        acs_gdf[['ASUTE004', 'ASUTE011']].sum(axis=1) / acs_gdf['ASUTE001']
    )
    two_or_more_cols = [
        'ASUTE005', 'ASUTE006', 'ASUTE007', 'ASUTE008',  # Owner: 2+
        'ASUTE012', 'ASUTE013', 'ASUTE014', 'ASUTE015'   # Renter: 2+
    ]
    acs_gdf['Pct_two_or_more_vehicles'] = acs_gdf[two_or_more_cols].sum(axis=1) / acs_gdf['ASUTE001']

    acs_gdf['Car_free_vulnerability'] = acs_gdf['Pct_no_vehicle'] * acs_gdf['Pct_below_poverty']

    acs_gdf['Active_mode_score'] = (
        (acs_gdf['ASORE018'] + acs_gdf['ASORE019']) / acs_gdf['ASORE001']  # walk + bike mode share
    ) * acs_gdf['Pct_short_commute'] # short commutes

    acs_gdf['Low_income_short_commute'] = (
        acs_gdf['Pct_below_poverty'] * acs_gdf['Pct_short_commute']
    )

    acs_bg_for_clip = acs_gdf[['Total_pop', 'geometry']].copy()  # still EPSG:3857 here
    acs_bg_for_clip['bg_area_m2'] = acs_bg_for_clip.geometry.area

    acs_gdf = acs_gdf.to_crs("EPSG:4326")
    acs_gdf = acs_gdf[['Total_pop', 'Pop_density', 'Pct_working_age', 'Child_dependency_ratio', 'Elderly_dependency_ratio',
                       'Pct_school_age', 'White_to_others_ratio', 'Pct_drive_alone', 'Pct_carpool', 
                       'Pct_public_transit', 'Pct_walk', 'Pct_bike', 'Pct_early_morning', 'Pct_peak_departure',
                       'Pct_evening', 'Pct_short_commute', 'Pct_moderate_commute', 'Pct_long_commute', 
                       'Pct_super_commuters', 'Pct_hs_or_higher', 'Pct_bachelors_or_higher', 'Pct_less_than_hs',
                       'Pct_below_poverty', 'Pct_near_poverty', 'Pct_above_2x_poverty', 'Per_capita_income', 
                       'Housing_unit_density', 'Pct_occupied_houses', 'Pct_no_vehicle', 'Pct_one_vehicle', 
                       'Pct_two_or_more_vehicles', 'Car_free_vulnerability', 'Active_mode_score', 
                       'Low_income_short_commute', 'geometry']]

    df['geometry'] = df.apply(lambda row: shapely.geometry.Point(row['Longitude'], row['Latitude']), axis=1)
    geo_df = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

    geo_df = gpd.sjoin(geo_df, acs_gdf, how="left", predicate="intersects")

    pts_3857 = geo_df.to_crs(epsg=3857)
    buffers = pts_3857.geometry.buffer(402)
    buffers_gdf = gpd.GeoDataFrame(geometry=buffers, crs=acs_gdf.to_crs(epsg=3857).crs)

    acs_3857 = acs_bg_for_clip
    clipped = gpd.overlay(acs_3857, buffers_gdf.reset_index(), how="intersection")

    if clipped.empty:
        geo_df['Pop_within_402m'] = 0
    else:
        clipped['intersect_area_m2'] = clipped.geometry.area
        clipped['frac_of_bg'] = clipped['intersect_area_m2'] / clipped['bg_area_m2']
        clipped['pop_in_buffer'] = clipped['Total_pop'] * clipped['frac_of_bg']

        pop_sums = (clipped.groupby('index')['pop_in_buffer']
                              .sum()
                              .round()
                              .astype(int)
                              .rename('Pop_within_402m'))

        geo_df['Pop_within_402m'] = geo_df.index.map(pop_sums.to_dict()).fillna(0).astype(int)

    geo_df = geo_df.drop(columns=['geometry', 'index_right'], errors='ignore')

    def base_name(c):
        if c.endswith('_left'):
            return c[:-5]
        if c.endswith('_right'):
            return c[:-6]
        return c

    df = df.copy()
    for col in geo_df.columns:
        if col in orig_cols:
            continue
        if base_name(col) in orig_cols:
            continue
        df[col] = geo_df[col]

    return df

In [52]:
process_static_data(
    processing_function=add_acs_data,
    feature_list=[]
)

### PAG POI

In [12]:
from geopy.distance import distance
import pandas as pd

def count_clusters_within_radius(cluster_data, lat, lon, radius_miles=0.25):
    nearby_clusters = cluster_data[
        cluster_data.apply(
            lambda row: distance((lat, lon), (row['Lat'], row['Lon'])).miles <= radius_miles,
            axis=1
        )
    ]

    cluster_counts = nearby_clusters['Cluster'].value_counts().to_dict()

    emp_by_cluster = (
        nearby_clusters.groupby('Cluster')['Emp']
        .sum()
        .astype(int)
        .to_dict()
    )

    total_employees = int(nearby_clusters['Emp'].sum())

    return cluster_counts, emp_by_cluster, total_employees


def add_pag_poi(df, feature_list, temp_output_path=None):
    pag_clusters_df = pd.read_excel(pag_poi_file)
    df = df.copy()

    if temp_output_path and os.path.exists(temp_output_path):
        existing = pd.read_csv(temp_output_path)
        completed_ids = set(existing['DeviceID'])
        unique_results = existing.to_dict("records")
    else:
        completed_ids = set()
        unique_results = []

    for idx, row in df.drop_duplicates(subset='DeviceID').iterrows():
        device_id, lat, lon = row['DeviceID'], row['Latitude'], row['Longitude']
        if pd.notnull(lat) and pd.notnull(lon) and device_id not in completed_ids:

            cluster_counts, emp_by_cluster, total_employees = count_clusters_within_radius(pag_clusters_df, lat, lon)

            results = {
                'DeviceID': device_id,
                'TotalEmployees_WithinRadius': total_employees
            }

            for cluster, count in cluster_counts.items():
                results[f'{cluster}'] = count

            for cluster, emp_count in emp_by_cluster.items():
                results[f'Emp_{cluster}'] = emp_count

            unique_results.append(results)

            if temp_output_path:
                pd.DataFrame(unique_results).fillna(0).to_csv(temp_output_path, index=False)

            print(f"✓ Completed DeviceID {device_id}")

    if unique_results:
        results_df = (
            pd.DataFrame(unique_results)
            .fillna(0)
            .astype(int, errors='ignore')
        )

        df = (
            df.merge(results_df, on='DeviceID', how='left')
            .fillna(0)
            .reset_index(drop=True)
        )

    return df

In [13]:
process_static_data(
    processing_function=add_pag_poi,
    feature_list=[],
    temp_output_path = "pag_poi_checkpoint.csv"
)

## Temporal Attributes

### Weather Data

In [62]:
from datetime import timedelta

import meteostat


def add_weather_data(df, feature_list):
    """
    Fetches weather data for the given dataset and dynamically adds specified features.

    Parameters:
    - df (pd.DataFrame): The dataset containing 'Latitude', 'Longitude', and 'DateHour'.
    - feature_list (list): List of weather attributes to extract (e.g., ['temp', 'wspd']).

    Returns:
    - pd.DataFrame: Updated dataset with requested weather attributes.
    """
    if 'Latitude' not in df.columns or 'Longitude' not in df.columns or 'DateHour' not in df.columns:
        raise ValueError("Dataset must contain 'Latitude', 'Longitude', and 'DateHour' columns.")

    for index, row in df.iterrows():
        lat, lon, datehour = row["Latitude"], row["Longitude"], row["DateHour"]

        if pd.notnull(lat) and pd.notnull(lon) and pd.notnull(datehour):
            try:
                date_time = pd.to_datetime(datehour)
                start_date = date_time
                end_date = start_date + timedelta(hours=1)
                weather_data = Hourly(meteostat.Point(lat,lon), start_date, end_date).fetch()

                if not weather_data.empty:
                    for feature in feature_list:
                        if feature in weather_data.columns:
                            df.at[index, feature] = weather_data.iloc[0].get(feature, None)

            except Exception as e:
                print(f"Error fetching weather for lat {lat}, lon {lon} at {date_time}: {e}")

    if 'prcp' in feature_list and 'prcp' in df.columns:
        df['prcp'].fillna(0, inplace=True)
        df['rain'] = df['prcp'] != 0
        df.drop(columns=['prcp'], inplace=True)

    return df

In [61]:
process_static_data(
    processing_function=add_weather_data,
    feature_list=['temp', 'prcp']
)

7
7


## Data Processing

### Generate Base Data

In [ ]:
def combine_push_count():
    combined_data = pd.DataFrame()

    for file_name in os.listdir(push_count_folder):
        if file_name.startswith("pedestrian_count_comparison") and file_name.endswith(".csv"):
            file_path = os.path.join(push_count_folder, file_name)
            data = pd.read_csv(file_path)
            combined_data = pd.concat([combined_data, data], ignore_index=True)

    return combined_data

In [ ]:
def process_data_step(filename, base_file, processing_function, feature_list):
    """
    Handles dataset reading, processing, and saving.

    Parameters:
    - filename (str): The path to the dataset file.
    - base_file (str): The path to the base dataset file (if the dataset doesn't exist).
    - processing_function (function): Function that modifies the dataset.
    - feature_list (list): List of attributes to extract.

    Returns:
    - None (saves the updated dataset to `filename`).
    """
    if os.path.exists(filename):
        df = pd.read_csv(filename, parse_dates=['DateHour'])
    else:
        df = pd.read_csv(base_file, parse_dates=['DateHour'])

    df = processing_function(df, feature_list)

    df.to_csv(filename, index=False)
    print(f"Updated dataset saved to {filename}")

### Merge

In [ ]:
import pandas as pd

def merge_new_features(csv_main_path, csv_features_path, output_csv_path):
    """
    Merges new features from csv_features_path into csv_main_path using 'DeviceID' as key.
    Does not overwrite existing columns in the main CSV.
    
    Parameters:
    - csv_main_path (str): Path to the main CSV file.
    - csv_features_path (str): Path to the CSV with additional features.
    - output_csv_path (str): Path to save the merged result.
    """

    df_main = pd.read_csv(csv_main_path)
    df_features = pd.read_csv(csv_features_path)


    new_cols = [col for col in df_features.columns if col != 'DeviceID' and col not in df_main.columns]

    if not new_cols:
        df_main.to_csv(output_csv_path, index=False)
        print("No new columns to merge. Original file saved.")
        return

    df_merged = df_main.merge(df_features[['DeviceID'] + new_cols], on='DeviceID', how='left')

    df_merged.to_csv(output_csv_path, index=False)

In [ ]:
merge_new_features(old_output_file, device_output_file, old_output_file)

In [ ]:
folder_path = 'Input/maxview_waiting_time'  

csv_files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

combined_df = pd.concat(
    [pd.read_csv(os.path.join(folder_path, file)) for file in csv_files],
    ignore_index=True
).drop(['Total_Counts', 'Counts'], axis=1)

combined_df = combined_df.drop_duplicates(['DateHour', 'DeviceID'])

combined_df.to_csv(queue_times_file, index=False)

In [ ]:
import numpy as np

ped_volume_data = pd.read_csv(ped_volume_file)
maxview_data = pd.read_csv(maxview_file)
features_data = pd.read_csv(device_output_file)
waiting_times_data = pd.read_csv(queue_times_file)

for df in [ped_volume_data, maxview_data, features_data, waiting_times_data]:
    df['DateHour'] = pd.to_datetime(df['DateHour'])

merged_df = pd.merge(ped_volume_data, maxview_data, on=['DateHour', 'DeviceID'], how='left')
merged_df = pd.merge(merged_df, features_data, on=['DateHour', 'DeviceID'], how='left')
merged_df = pd.merge(merged_df, waiting_times_data, on=['DateHour', 'DeviceID'], how='left')

merged_df.insert(3, 'Total_Push', merged_df['NS_Push_Total'] + merged_df['EW_Push_Total'])

merged_df.loc[merged_df['AvgSpeed_y'] == 0, 'AvgSpeed_y'] = np.inf

columns_to_clean = ['Per_capita_income', 'AvgCycleLen_s', 'White_to_others_ratio',
                    'D4A','D4B025','D4B050','D4C','D4D','D4E', 'AvgSpeed_y',
                    'D5DR','D5DRI','D5DE','D5DEI', 'Maximum_Waiting_Time', 'Average_Waiting_Time']

for col in columns_to_clean:

    merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')

    merged_df[col] = merged_df[col].replace([np.inf, -np.inf], np.nan)

    merged_df[f'{col}_missing'] = (
        merged_df[col].isna() | (merged_df[col] < 0)
    ).astype(int)

    merged_df[col] = merged_df[col].mask(merged_df[col] < 0, np.nan)

    median_value = merged_df[col].median()
    merged_df[col] = merged_df[col].fillna(median_value)

merged_df.columns = merged_df.columns.str.replace('NS', 'Major').str.replace('EW', 'Minor')

merged_df.to_csv(final_output_file, index=False)

In [ ]:
dup_counts = combined_df.groupby(['DeviceID', 'DateHour']).size()

dup_keys = dup_counts[dup_counts > 1]

print("Number of duplicated key combinations:", len(dup_keys))
print(dup_keys.head())

Number of duplicated key combinations: 0
Series([], dtype: int64)


In [ ]:
folder_path = 'Input/maxview_waiting_time'  # <- change this to your actual folder path

# Get a list of all CSV files in the folder
csv_files = [file for file in os.listdir(folder_path) if file.startswith('maxview_ped') and file.endswith('.csv')]

# Combine all CSV files
combined_df = pd.concat(
    [pd.read_csv(os.path.join(folder_path, file)) for file in csv_files],
    ignore_index=True
)

combined_df = combined_df.drop_duplicates(['DeviceID', 'DateHour'])
combined_df = combined_df.drop(['Total_Counts', 'Counts'], axis=1)

# Save the cleaned DataFrame back to a new CSV
combined_df.to_csv(queue_times_file, index=False)

In [ ]:
folder_path = 'Input/Maxview'  # <- change this to your actual folder path

# Get a list of all CSV files in the folder
csv_files = [file for file in os.listdir(folder_path) if file.startswith('Maxview_data') and file.endswith('.csv')]

# Combine all CSV files
combined_df = pd.concat(
    [pd.read_csv(os.path.join(folder_path, file)) for file in csv_files],
    ignore_index=True
)

combined_df = combined_df.drop_duplicates(['DeviceID', 'DateHour'])

# Save the cleaned DataFrame back to a new CSV
combined_df.to_csv(maxview_file, index=False)

In [ ]:
import numpy as np
import pandas as pd

def feature_issue_summary(df, outlier_method='iqr', z_thresh=3.0):
    """
    Returns a DataFrame summarising, for every column:
    - NaN count
    - Negative-value count (numeric only)
    - ±Inf count (numeric only)
    - Outlier count  (IQR or z-score)
    """
    summary = []

    for col in df.columns:
        col_data = df[col]

        if pd.api.types.is_numeric_dtype(col_data) and not pd.api.types.is_bool_dtype(col_data):
            # NaNs, negatives, infinities
            n_nan  = col_data.isna().sum()
            n_neg  = (col_data < 0).sum()
            n_inf  = np.isinf(col_data).sum()          # counts both +Inf and -Inf

            # Outliers
            n_outliers = 0
            non_na = col_data.replace([np.inf, -np.inf], np.nan).dropna()
            if not non_na.empty:
                try:
                    if outlier_method == 'zscore':
                        z_scores = (non_na - non_na.mean()) / non_na.std()
                        n_outliers = (z_scores.abs() > z_thresh).sum()
                    else:  # IQR
                        q1, q3 = non_na.quantile([0.25, 0.75])
                        iqr = q3 - q1
                        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
                        n_outliers = ((non_na < lo) | (non_na > hi)).sum()
                except Exception as e:
                    print(f"Skipping outlier check for '{col}': {e}")

            summary.append({
                'Feature'  : col,
                'NaNs'     : n_nan,
                'Negatives': n_neg,
                'Infs'     : n_inf,
                'Outliers' : n_outliers
            })

        else:
            # non-numeric / boolean
            summary.append({
                'Feature'  : col,
                'NaNs'     : col_data.isna().sum(),
                'Negatives': 'N/A',
                'Infs'     : 'N/A',
                'Outliers' : 'N/A'
            })

    return (pd.DataFrame(summary)
              .sort_values(by='NaNs', ascending=False)
              .reset_index(drop=True))

In [ ]:
feature_cols = [col for col in merged_df.columns if col not in exclude_cols]
features_only_df = merged_df[feature_cols]

In [ ]:
summary_df = feature_issue_summary(features_only_df)
summary_df

,Feature,NaNs,Negatives,Infs,Outliers
0,Major_Max_speed_mph,297,0,0,0
1,Minor_Max_speed_mph,242,0,0,127
2,treecanopy,116,0,0,35
3,tc_goal,116,0,0,0
4,priority_i,116,0,0,0
...,...,...,...,...,...
255,D1C5_SVC,0,0,0,142
256,D1C5_ENT,0,0,0,151
257,D1C8_RET,0,0,0,128
258,D1C8_OFF,0,0,0,118
